** NUMBA PROGRAMMING **

1. You are given a large NumPy array of size 5000000 initialized with random
values. Compute the following element-wise operation: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance difference of CPU with GPU.
a. Modify the kernel to use float32 and float64

In [5]:
import numpy as np
import time
from numba import cuda
# CUDA Kernel
@cuda.jit
def compute_kernel(x, y):

    i = cuda.grid(1)

    if i < x.size:
        y[i] = x[i]*x[i] + 3*x[i] + 5
n = 5000000


# -------- FLOAT32 --------
print("----- FLOAT32 RESULTS -----")

x = np.random.rand(n).astype(np.float32)
y = np.zeros(n, dtype=np.float32)

threads = 256
blocks = (n + threads - 1) // threads


# CPU computation
start = time.time()

cpu_result = x*x + 3*x + 5

cpu_time32 = time.time() - start

print("CPU Time (float32):", cpu_time32)


# GPU computation
start = time.time()

compute_kernel[blocks, threads](x, y)

cuda.synchronize()

gpu_time32 = time.time() - start

print("GPU Time (float32):", gpu_time32)

# -------- FLOAT64 --------
print("\n----- FLOAT64 RESULTS -----")

x = np.random.rand(n).astype(np.float64)
y = np.zeros(n, dtype=np.float64)

# CPU computation
start = time.time()

cpu_result = x*x + 3*x + 5

cpu_time64 = time.time() - start

print("CPU Time (float64):", cpu_time64)

# GPU computation
start = time.time()

compute_kernel[blocks, threads](x, y)

cuda.synchronize()

gpu_time64 = time.time() - start

print("GPU Time (float64):", gpu_time64)

----- FLOAT32 RESULTS -----
CPU Time (float32): 0.014655351638793945
GPU Time (float32): 0.11544489860534668

----- FLOAT64 RESULTS -----


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


CPU Time (float64): 0.030658960342407227
GPU Time (float64): 0.10605907440185547


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


2. Implement and benchmark a 1-D histogram computa8on for 1 million
random values in Python using Numba. Compare diﬀerent approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness.

In [6]:
import numpy as np
import time
from numba import njit
n = 1000000
data = np.random.rand(n)
bins = 10
#PURE PYTHON
def histogram_python(data, bins):
    hist = [0] * bins
    for value in data:
        index = int(value * bins)
        if index == bins:
            index = bins - 1
        hist[index] += 1
    return hist
start = time.time()
hist_py = histogram_python(data, bins)
python_time = time.time() - start
print("Python Histogram Time:", python_time)

# NUMPY
start = time.time()
hist_np, edges = np.histogram(data, bins=bins)
numpy_time = time.time() - start
print("NumPy Histogram Time:", numpy_time)

#NUMBA
@njit
def histogram_numba(data, bins):

    hist = np.zeros(bins)

    for i in range(len(data)):

        index = int(data[i] * bins)

        if index == bins:
            index = bins - 1

        hist[index] += 1

    return hist


start = time.time()

hist_nb = histogram_numba(data, bins)

numba_time = time.time() - start

print("Numba Histogram Time:", numba_time)

print("\nHistogram Values (Python):", hist_py)
print("Histogram Values (NumPy):", hist_np)
print("Histogram Values (Numba):", hist_nb)

Python Histogram Time: 0.2571728229522705
NumPy Histogram Time: 0.018463611602783203
Numba Histogram Time: 1.078312873840332

Histogram Values (Python): [99954, 100241, 99684, 100212, 100003, 100203, 99916, 99384, 100526, 99877]
Histogram Values (NumPy): [ 99954 100241  99684 100212 100003 100203  99916  99384 100526  99877]
Histogram Values (Numba): [ 99954. 100241.  99684. 100212. 100003. 100203.  99916.  99384. 100526.
  99877.]


The pure Python implementation was the slowest due to interpreted execution.

3. Write a func8on monte_carlo_pi(nsamples) that es8mates the value of π by
genera8ng random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).
a. Implement the func8on in pure Python first and later create a Numba
version.
b. Program a script to compare the execu8on 8me for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).
c. Why does the very first execu8on of the Numba func8on take slightly
longer than the second execu8on?

In [7]:
import random
import numpy as np
import time
from numba import njit

#python
def monte_carlo_pi_python(nsamples):

    inside = 0

    for i in range(nsamples):

        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside += 1

    pi_estimate = 4 * inside / nsamples
    return pi_estimate
# NUMBA VERSION
@njit
def monte_carlo_pi_numba(nsamples):

    inside = 0

    for i in range(nsamples):

        x = np.random.random()
        y = np.random.random()

        if x*x + y*y < 1:
            inside += 1

    pi_estimate = 4 * inside / nsamples
    return pi_estimate
samples = 5000000
#PYTHON EXECUTION
start = time.time()

pi_python = monte_carlo_pi_python(samples)

python_time = time.time() - start

print("Python π estimate:", pi_python)
print("Python Time:", python_time)
#NUMBA EXECUTION
start = time.time()
pi_numba = monte_carlo_pi_numba(samples)
numba_time = time.time() - start
print("\nNumba π estimate:", pi_numba)
print("Numba Time:", numba_time)
# SPEEDUP
speedup = python_time / numba_time

print("\nSpeedup Factor:", speedup)

Python π estimate: 3.1415776
Python Time: 1.0003204345703125

Numba π estimate: 3.141436
Numba Time: 0.1932220458984375

Speedup Factor: 5.177051251678117


4. You have a 1D NumPy array represen8ng pixel intensi8es (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.
a. Write a func8on adjust_brightness(pixel_value) using the @vectorize
decorator.
b. Apply this func8on to an array of 10 million random integers.
c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the 8me diﬀerence when the work is automa8cally split
across your CPU cores.
d. What happens if you try to pass a list instead of a NumPy array to this
func8on?

In [8]:
import numpy as np
import time
from numba import vectorize


# NORMAL VECTORIZE VERSION
@vectorize
def adjust_brightness(pixel_value):

    new_value = pixel_value * 1.2

    if new_value > 255:
        new_value = 255

    return int(new_value)
# generate 10 million pixel values
pixels = np.random.randint(0, 256, 10000000)
start = time.time()
result = adjust_brightness(pixels)
time_normal = time.time() - start
print("Time (vectorize):", time_normal)
# PARALLEL VERSION
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel_value):

    new_value = pixel_value * 1.2

    if new_value > 255:
        new_value = 255

    return int(new_value)
start = time.time()
result_parallel = adjust_brightness_parallel(pixels)
time_parallel = time.time() - start
print("Time (parallel vectorize):", time_parallel)

Time (vectorize): 0.11007118225097656
Time (parallel vectorize): 0.029839754104614258


5. Write Python code to generate synthe8c training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logis8c regression
using the mathema8cal formula for gradient descent:
a. Using standard NumPy (without Numba)
b. Using Numba JIT accelera8on
c. Compare correctness and performance.

In [9]:
import numpy as np
import time
from numba import njit
samples = 100000
features = 10

X = np.random.randn(samples, features)
y = np.random.choice([-1, 1], samples)
learning_rate = 0.01
epochs = 50
#  NUMPY IMPLEMENTATION
def logistic_regression_numpy(X, y, lr, epochs):
    n, m = X.shape
    weights = np.zeros(m)
    for epoch in range(epochs):
        z = X @ weights
        gradient = -(y * X.T) / (1 + np.exp(y * z))
        weights = weights - lr * gradient.mean(axis=1)
    return weights
start = time.time()
weights_numpy = logistic_regression_numpy(X, y, learning_rate, epochs)
numpy_time = time.time() - start
print("NumPy Training Time:", numpy_time)

# NUMBA IMPLEMENTATION
@njit
def logistic_regression_numba(X, y, lr, epochs):

    n, m = X.shape
    weights = np.zeros(m)
    for epoch in range(epochs):
        for i in range(n):
            z = 0.0
            for j in range(m):
                z += X[i][j] * weights[j]
            grad = -y[i] / (1 + np.exp(y[i] * z))
            for j in range(m):
                weights[j] -= lr * grad * X[i][j]
    return weights
start = time.time()
weights_numba = logistic_regression_numba(X, y, learning_rate, epochs)
numba_time = time.time() - start
print("Numba Training Time:", numba_time)
print("\nWeights (NumPy):", weights_numpy[:5])
print("Weights (Numba):", weights_numba[:5])

NumPy Training Time: 0.7292656898498535
Numba Training Time: 0.4685325622558594

Weights (NumPy): [ 0.00088316 -0.0007284  -0.00081048 -0.00027375  0.00028686]
Weights (Numba): [ 0.00949907 -0.09390935  0.00461942  0.01487081 -0.05102644]


6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [10]:
import numpy as np
from numba import cuda
import time
#  CUDA Kernel
@cuda.jit
def matrix_add(A, B, C):

    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        C[row][col] = A[row][col] + B[row][col]
N = 1024
A = np.random.rand(N, N)
B = np.random.rand(N, N)

C = np.zeros((N, N))


# threads per block
threads_per_block = (16, 16)

# blocks per grid
blocks_per_grid_x = (N + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (N + threads_per_block[1] - 1) // threads_per_block[1]

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

start = time.time()
matrix_add[blocks_per_grid, threads_per_block](A, B, C)
cuda.synchronize()
end = time.time()
print("GPU Matrix Addition Time:", end - start)
print("\nSample Output (first 5 values):")
print(C[:2, :5])

GPU Matrix Addition Time: 0.11575174331665039

Sample Output (first 5 values):
[[1.16051037 0.66823051 0.48648307 1.22797147 0.86761985]
 [0.15891406 1.24001555 0.49633745 0.83477428 1.08801304]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
